In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pybromo as pbm
import os, time, tables
# directory to save HDF5
folder = './Pybromo_simulations'
os.makedirs(folder, exist_ok=True)

## System setup: **BOX** & **LASER**

In [ ]:
# grant reproducibility by defining random state
rs = np.random.RandomState(seed=1)
# simulate box system
box = pbm.Box(
    x1 = -2.5e-6, x2 = +2.5e-6,
    y1 = -2.5e-6, y2 = +2.5e-6,
    z1 = -5.0e-6, z2 = +5.0e-6
)
# PSF for laser source centering the excitation confocal volume
w0 = 0.25e-6 
z0 = 1.0e-6
# conversion radius -> sigma for Gaussian
sx = w0 / np.sqrt(2)
sy = w0 / np.sqrt(2)
sz = z0 / np.sqrt(2)

# creation of laser PSF
psf = pbm.GaussianPSF(
    xc=0, yc=0, zc=0,  # Box centering
    sx=sx, sy=sy, sz=sz
)

### Absolute number of molecules

1. **Convert concentration to molarity**  
   `C [mol/L] = C [g/L] / MW [g/mol]`

2. **Calculate the confocal volume**  
   `V_conf = (4/3) × π × r_x × r_y × r_z`

3. **Calculate the mean number of molecules in the confocal volume**  
   `N_conf = C × N_A × V_conf`

4. **Convert to the absolute number of molecules in the simulation box**  
   `N_box = N_conf × (V_box / V_conf)`

5. **Round to the nearest integer** to obtain the number of particles used in the simulation.

# I. **free EtBr** run

In [ ]:
# diffusion specs
EtBr_Du = 425.0          # µm²/s
EtBr_D = EtBr_Du*(1e-6)**2   # m²/s
# average number of molecules in confocal volume
n_avg = 1
# ellissoid confocal volume
vol_psf = (4/3) * np.pi * (w0**2) * z0
# excited volume
vol_box = (box.x2 - box.x1) * (box.y2 - box.y1) * (box.z2 - box.z1)
# absolute number of particles to generate in the box 
n_particles = int(n_avg * (vol_box / vol_psf))
# time simulation parameters
t_step = 5e-6   # time step (s), t
t_max = 60       # total simulation (s)

print(f"To have {n_avg} molecules in the excitation volume,")
print(f"There will be needed {n_particles} particles in the box.")

In [ ]:
# generate particles in the simulation
P = pbm.Particles(
    num_particles = n_particles, D = EtBr_D, box = box, rs = rs
)
# prepare simulation timing before running it
S = pbm.ParticlesSimulation(
    t_step=t_step, t_max=t_max, particles=P, box=box, psf=psf
)
# display expected size (general memory information)
print(S.print_sizes())

In [ ]:
# execute simulation 
S.simulate_diffusion(
    save_pos=False,           # <-- True if we want to record and save positions
    total_emission=False,    # <-- False if we want to generate timestamps
    rs=rs,
    chunksize=2**19,
    chunkslice='times',
    path = folder
)
print('Shape emission:', S.emission.shape)  #expected: (n_particles, n_timepoints)

In [ ]:
S.store.close()

# II. **EtBr-DNA** run (0.004 molecules in confocal vol.)

In [ ]:
# diffusion specs
EtBr_DNA_Du = 122.0          # µm²/s
EtBr_DNA_D = EtBr_DNA_Du*(1e-6)**2   # m²/s
# average number of molecules in confocal volume
n_avg_EtBr_DNA = 0.004

# diffusion specs
EtBr_free_Du = 425.0          # µm²/s
EtBr_free_D = EtBr_free_Du*(1e-6)**2   # m²/s
# average number of molecules in confocal volume
n_avg_EtBr_free = 1 - n_avg_EtBr_DNA

# ellissoid confocal volume
vol_psf = (4/3) * np.pi * (w0**2) * z0
# excited volume
vol_box = (box.x2 - box.x1) * (box.y2 - box.y1) * (box.z2 - box.z1)
# absolute number of particles to generate in the box 
N_EtBr_DNA = int(n_avg_EtBr_DNA * (vol_box / vol_psf))
N_EtBr_free = int(n_avg_EtBr_free * (vol_box / vol_psf))
# time simulation parameters
t_step = 1e-6   # time step (s), t
t_max = 1       # total simulation (s)

print(f'EtBr-free specs:\n- Diffusion: {EtBr_free_D} m^2/s\n- Avg in confocal volume: {n_avg_EtBr_free}\n- Absolute number: {N_EtBr_free}')
print(f'EtBr-DNA  specs:\n- Diffusion: {EtBr_DNA_D} m^2/s\n- Avg in confocal volume: {n_avg_EtBr_DNA}\n- Absolute number: {N_EtBr_DNA}')

In [ ]:
# generate EtBr free molecules in sim
P = pbm.Particles(
    num_particles = N_EtBr_free, D = EtBr_free_D, box = box, rs = rs
)
# add EtBr-DNA bind molecules in sim
P.add(
    num_particles = N_EtBr_DNA, D = EtBr_DNA_D
)
# prepare simulation timing before running it
S = pbm.ParticlesSimulation(
    t_step=t_step, t_max=t_max, particles=P, box=box, psf=psf
)
# display expected size (general memory information)
print(S.print_sizes())

In [ ]:
# execute simulation 
S.simulate_diffusion(
    save_pos=False,           # <-- True if we want to record and save positions
    total_emission=False,    # <-- False if we want to generate timestamps
    rs=rs,
    chunksize=2**19,
    chunkslice='times',
    path = folder
)
print('Shape emission:', S.emission.shape)  #expected: (n_particles, n_timepoints)

In [ ]:
S.store.close()

# II. **EtBr-DNA** run (0.00905 molecules in confocal vol.)

In [ ]:
# diffusion specs
EtBr_DNA_Du = 122.0          # µm²/s
EtBr_DNA_D = EtBr_DNA_Du*(1e-6)**2   # m²/s
# average number of molecules in confocal volume
n_avg_EtBr_DNA = 0.00905

# diffusion specs
EtBr_free_Du = 425.0          # µm²/s
EtBr_free_D = EtBr_free_Du*(1e-6)**2   # m²/s
# average number of molecules in confocal volume
n_avg_EtBr_free = 1 - n_avg_EtBr_DNA

# ellissoid confocal volume
vol_psf = (4/3) * np.pi * (w0**2) * z0
# excited volume
vol_box = (box.x2 - box.x1) * (box.y2 - box.y1) * (box.z2 - box.z1)
# absolute number of particles to generate in the box 
N_EtBr_DNA = int(n_avg_EtBr_DNA * (vol_box / vol_psf))
N_EtBr_free = int(n_avg_EtBr_free * (vol_box / vol_psf))
# time simulation parameters
t_step = 1e-6   # time step (s), t
t_max = 1       # total simulation (s)

print(f'EtBr-free specs:\n- Diffusion: {EtBr_free_D} m^2/s\n- Avg in confocal volume: {n_avg_EtBr_free}\n- Absolute number: {N_EtBr_free}')
print(f'EtBr-DNA  specs:\n- Diffusion: {EtBr_DNA_D} m^2/s\n- Avg in confocal volume: {n_avg_EtBr_DNA}\n- Absolute number: {N_EtBr_DNA}')

In [ ]:
# generate EtBr free molecules in sim
P = pbm.Particles(
    num_particles = N_EtBr_free, D = EtBr_free_D, box = box, rs = rs
)
# add EtBr-DNA bind molecules in sim
P.add(
    num_particles = N_EtBr_DNA, D = EtBr_DNA_D
)
# prepare simulation timing before running it
S = pbm.ParticlesSimulation(
    t_step=t_step, t_max=t_max, particles=P, box=box, psf=psf
)
# display expected size (general memory information)
print(S.print_sizes())

In [ ]:
# execute simulation 
S.simulate_diffusion(
    save_pos=False,           # <-- True if we want to record and save positions
    total_emission=False,    # <-- False if we want to generate timestamps
    rs=rs,
    chunksize=2**19,
    chunkslice='times',
    path = folder
)
print('Shape emission:', S.emission.shape)  #expected: (n_particles, n_timepoints)

In [ ]:
S.store.close()

# II. **EtBr-DNA** run (0.0947 molecules in confocal vol.)

In [ ]:
# diffusion specs
EtBr_DNA_Du = 122.0          # µm²/s
EtBr_DNA_D = EtBr_DNA_Du*(1e-6)**2   # m²/s
# average number of molecules in confocal volume
n_avg_EtBr_DNA = 0.0947

# diffusion specs
EtBr_free_Du = 425.0          # µm²/s
EtBr_free_D = EtBr_free_Du*(1e-6)**2   # m²/s
# average number of molecules in confocal volume
n_avg_EtBr_free = 1 - n_avg_EtBr_DNA

# ellissoid confocal volume
vol_psf = (4/3) * np.pi * (w0**2) * z0
# excited volume
vol_box = (box.x2 - box.x1) * (box.y2 - box.y1) * (box.z2 - box.z1)
# absolute number of particles to generate in the box 
N_EtBr_DNA = int(n_avg_EtBr_DNA * (vol_box / vol_psf))
N_EtBr_free = int(n_avg_EtBr_free * (vol_box / vol_psf))
# time simulation parameters
t_step = 1e-6   # time step (s), t
t_max = 1       # total simulation (s)

print(f'EtBr-free specs:\n- Diffusion: {EtBr_free_D} m^2/s\n- Avg in confocal volume: {n_avg_EtBr_free}\n- Absolute number: {N_EtBr_free}')
print(f'EtBr-DNA  specs:\n- Diffusion: {EtBr_DNA_D} m^2/s\n- Avg in confocal volume: {n_avg_EtBr_DNA}\n- Absolute number: {N_EtBr_DNA}')

In [ ]:
# generate EtBr free molecules in sim
P = pbm.Particles(
    num_particles = N_EtBr_free, D = EtBr_free_D, box = box, rs = rs
)
# add EtBr-DNA bind molecules in sim
P.add(
    num_particles = N_EtBr_DNA, D = EtBr_DNA_D
)
# prepare simulation timing before running it
S = pbm.ParticlesSimulation(
    t_step=t_step, t_max=t_max, particles=P, box=box, psf=psf
)
# display expected size (general memory information)
print(S.print_sizes())

In [ ]:
# execute simulation 
S.simulate_diffusion(
    save_pos=False,           # <-- True if we want to record and save positions
    total_emission=False,    # <-- False if we want to generate timestamps
    rs=rs,
    chunksize=2**19,
    chunkslice='times',
    path = folder
)
print('Shape emission:', S.emission.shape)  #expected: (n_particles, n_timepoints)

In [ ]:
S.store.close()